In [1]:

# TECH CHALLENGE - FASE 1
# ETAPA 4 - Preparacao para Machine Learning

from pathlib import Path
import pandas as pd

# Caracteristicas morfologicas
caracteristicas = [
    "radius",
    "texture",
    "perimeter",
    "area",
    "smoothness",
    "compactness",
    "concavity",
    "concave_points",
    "symmetry",
    "fractal_dimension"
]

medidas = ["mean", "se", "worst"]

# Nomes das 32 colunas
colunas = ["id", "diagnosis"]

for medida in medidas:
    for caracteristica in caracteristicas:
        colunas.append(f"{caracteristica}_{medida}")

# Localizar o dataset
raiz = Path.cwd()

if raiz.name == "notebooks":
    raiz = raiz.parent

arquivo = raiz / "data" / "raw" / "wdbc.data"

if not arquivo.is_file():
    raise FileNotFoundError("Dataset nao encontrado!")

# Carregar a base
df_modelagem = pd.read_csv(
    arquivo,
    header=None,
    names=colunas
)

print("Dataset carregado!")
print("Dimensoes:", df_modelagem.shape)

display(df_modelagem.head())


Dataset carregado!
Dimensoes: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:

# ETAPA 4 - Separacao das variaveis X e y

print("=== PREPARACAO DOS DADOS ===")

# X contem apenas as 30 caracteristicas preditoras
X = df_modelagem.drop(
    columns=["id", "diagnosis"]
).copy()

# Converter os diagnosticos para numeros
mapa_diagnostico = {
    "B": 0,
    "M": 1
}

y = df_modelagem["diagnosis"].map(
    mapa_diagnostico
)

# Verificar se todos os diagnosticos foram convertidos
if y.isnull().any():
    raise ValueError(
        "Existem diagnosticos nao reconhecidos!"
    )

y = y.astype(int)

# Validar as dimensoes
assert X.shape == (569, 30)
assert y.shape == (569,)

print("\nDimensoes de X:", X.shape)
print("Dimensoes de y:", y.shape)

print("\nDistribuicao da variavel-alvo:")
print(y.value_counts().sort_index())

print("\nPrimeiras caracteristicas:")
display(X.head())

print("\nPrimeiros diagnosticos codificados:")
display(y.head())


=== PREPARACAO DOS DADOS ===

Dimensoes de X: (569, 30)
Dimensoes de y: (569,)

Distribuicao da variavel-alvo:
diagnosis
0    357
1    212
Name: count, dtype: int64

Primeiras caracteristicas:


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678



Primeiros diagnosticos codificados:


0    1
1    1
2    1
3    1
4    1
Name: diagnosis, dtype: int64

In [3]:

# ETAPA 4 - Divisao dos dados
# Treinamento: aproximadamente 70%
# Validacao: aproximadamente 15%
# Teste: aproximadamente 15%

from sklearn.model_selection import train_test_split

print("=== DIVISAO DOS DADOS ===")

# PRIMEIRA DIVISAO
# Separar 70% para treinamento e 30% temporarios

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# SEGUNDA DIVISAO
# Dividir os 30% temporarios em duas metades
# Uma para validacao e outra para teste

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\n=== DIMENSOES DOS CONJUNTOS ===")

print("Treinamento:", X_train.shape)
print("Validacao:", X_val.shape)
print("Teste:", X_test.shape)

print("\nTotal de registros:")
print(len(X_train) + len(X_val) + len(X_test))


=== DIVISAO DOS DADOS ===

=== DIMENSOES DOS CONJUNTOS ===
Treinamento: (398, 30)
Validacao: (85, 30)
Teste: (86, 30)

Total de registros:
569


In [4]:

# ETAPA 4 - Validacao da divisao dos dados

import pandas as pd

print("=== DISTRIBUICAO DAS CLASSES ===")

conjuntos = {
    "Original": y,
    "Treinamento": y_train,
    "Validacao": y_val,
    "Teste": y_test
}

resultados = []

for nome, classes in conjuntos.items():

    resultados.append({
        "Conjunto": nome,
        "Total": len(classes),
        "Benignos": (classes == 0).sum(),
        "Malignos": (classes == 1).sum(),
        "% Malignos": round(
            (classes == 1).mean() * 100, 2
        )
    })

tabela_divisao = pd.DataFrame(resultados)

display(tabela_divisao)

# Verificar se nao existem registros compartilhados

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

# Verificar se preservamos todas as amostras
assert (
    len(X_train) + len(X_val) + len(X_test)
) == len(X)

print("\nNenhum indice compartilhado entre os conjuntos.")
print("Todas as amostras foram preservadas.")


=== DISTRIBUICAO DAS CLASSES ===


,Conjunto,Total,Benignos,Malignos,% Malignos
0,Original,569,357,212,37.26
1,Treinamento,398,250,148,37.19
2,Validacao,85,53,32,37.65
3,Teste,86,54,32,37.21



Nenhum indice compartilhado entre os conjuntos.
Todas as amostras foram preservadas.


### Planejamento da separação dos conjuntos de dados

Para o desenvolvimento dos modelos supervisionados, foi definida uma estratégia de divisão estratificada do conjunto de dados, com proporções aproximadas de 70% para treinamento, 15% para validação e 15% para teste.

A estratificação foi escolhida para preservar aproximadamente a distribuição das classes benignas e malignas em cada conjunto.

Foi estabelecida a semente aleatória 42 para assegurar a reprodutibilidade da separação.

O conjunto de treinamento será utilizado para o ajuste dos modelos, o conjunto de validação para a comparação de configurações e seleção do modelo e o conjunto de teste para avaliação final.

As transformações de pré-processamento serão ajustadas exclusivamente com os dados apropriados de treinamento, evitando vazamento de informações entre as etapas de desenvolvimento e avaliação.

## Padronização das características numéricas

Nesta etapa, será utilizado o StandardScaler para
padronizar as 30 características preditoras.

Os parâmetros de padronização serão calculados
exclusivamente com os dados de treinamento.

O conjunto de validação receberá a mesma
transformação, sem um novo ajuste.

O conjunto de teste permanecerá reservado
para a avaliação final dos modelos.

In [5]:

# ETAPA 4 - Padronizacao das caracteristicas

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("=== PADRONIZACAO DOS DADOS ===")

# 1. Criar o pipeline de pre-processamento
pipeline_preprocessamento = Pipeline(
    steps=[
        ("padronizacao", StandardScaler())
    ]
)

# 2. Ajustar somente com dados de treinamento
# e transformar esses mesmos dados
X_train_pad = pipeline_preprocessamento.fit_transform(
    X_train
)

# 3. Aplicar a transformacao na validacao
# sem recalcular os parametros
X_val_pad = pipeline_preprocessamento.transform(
    X_val
)

# 4. Recuperar os nomes das caracteristicas
X_train_pad_df = pd.DataFrame(
    X_train_pad,
    columns=X_train.columns,
    index=X_train.index
)

# 5. Mostrar resultados
print("\nDimensoes do treinamento padronizado:")
print(X_train_pad_df.shape)

print("\nDimensoes da validacao padronizada:")
print(X_val_pad.shape)

print("\nPrimeiras linhas padronizadas:")
display(X_train_pad_df.head())

# 6. Verificar media e desvio-padrao
print("\nMedia aproximada das caracteristicas:")
print(np.round(X_train_pad.mean(axis=0)[:5], 6))

print("\nDesvio-padrao das caracteristicas:")
print(np.round(X_train_pad.std(axis=0)[:5], 6))

# 7. Validacao automatica
assert np.allclose(
    X_train_pad.mean(axis=0), 0, atol=1e-10
)

assert np.allclose(
    X_train_pad.std(axis=0), 1, atol=1e-10
)

print("\nPadronizacao validada com sucesso!")


=== PADRONIZACAO DOS DADOS ===

Dimensoes do treinamento padronizado:
(398, 30)

Dimensoes da validacao padronizada:
(85, 30)

Primeiras linhas padronizadas:


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
78,1.705820,1.049745,2.121295,1.669337,2.276458,4.544295,3.539139,2.848393,4.032390,2.639773,...,1.459510,0.949589,1.868176,1.306840,1.335383,2.313855,2.346427,2.049399,3.944281,0.866554
330,0.536388,-0.909945,0.569095,0.390316,-0.065868,0.644945,0.393543,0.557002,-0.116781,-0.401530,...,0.508294,-0.645315,0.501917,0.330087,0.477013,1.242091,1.052574,1.255850,0.167978,0.405242
378,-0.131456,-0.993336,-0.148848,-0.211543,-0.916169,-0.508574,-0.567530,-0.606645,-0.079867,-0.512454,...,-0.362451,-1.028485,-0.280415,-0.399387,-0.196219,0.368661,-0.080795,-0.140014,0.741988,0.687521
213,0.928077,1.418055,0.925405,0.828546,0.329734,0.223748,0.983188,0.443947,-1.866520,-0.555982,...,0.365921,0.351910,0.386082,0.243539,-0.330866,-0.464721,0.030311,-0.072253,-2.040715,-0.861167
89,0.144699,-0.972488,0.158315,-0.009697,1.205760,0.585041,0.137701,0.562858,1.116158,0.117991,...,0.008957,-1.257732,0.059367,-0.140450,-0.187804,0.359126,-0.064176,0.376471,0.373873,0.047726



Media aproximada das caracteristicas:
[ 0. -0.  0.  0. -0.]

Desvio-padrao das caracteristicas:
[1. 1. 1. 1. 1.]

Padronizacao validada com sucesso!


## Definição dos pipelines de classificação

Nesta etapa, serão preparados dois modelos de
Machine Learning supervisionado:

1. Regressão Logística com padronização.
2. Random Forest sem padronização.

Os modelos serão inicialmente configurados
sem treinamento. Os ajustes e a avaliação
serão realizados nas próximas etapas.

In [6]:

# ETAPA 4 - Criacao dos pipelines de classificacao

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("=== CONFIGURACAO DOS MODELOS ===")

# MODELO 1 - REGRESSAO LOGISTICA
# Padronizacao seguida de classificacao

pipeline_logistica = Pipeline(
    steps=[
        ("padronizacao", StandardScaler()),
        ("classificador", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

# MODELO 2 - RANDOM FOREST
# Arvores de decisao sem padronizacao

pipeline_floresta = Pipeline(
    steps=[
        ("classificador", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Organizar os modelos em um dicionario
modelos = {
    "Regressao Logistica": pipeline_logistica,
    "Random Forest": pipeline_floresta
}

# Exibir as etapas dos pipelines
for nome, modelo in modelos.items():

    print(f"\nModelo: {nome}")

    for etapa, componente in modelo.steps:
        print(f"  {etapa}: {componente}")

print("\nPipelines criados com sucesso!")
print("Os modelos ainda nao foram treinados.")


=== CONFIGURACAO DOS MODELOS ===

Modelo: Regressao Logistica
  padronizacao: StandardScaler()
  classificador: LogisticRegression(max_iter=2000, random_state=42)

Modelo: Random Forest
  classificador: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

Pipelines criados com sucesso!
Os modelos ainda nao foram treinados.


In [7]:

# ETAPA 4 - Verificacao dos pipelines

print("=== VALIDACAO DOS PIPELINES ===")

# Conferir os nomes das etapas
assert list(pipeline_logistica.named_steps) == [
    "padronizacao",
    "classificador"
]

assert list(pipeline_floresta.named_steps) == [
    "classificador"
]

# Conferir os classificadores
assert isinstance(
    pipeline_logistica.named_steps["classificador"],
    LogisticRegression
)

assert isinstance(
    pipeline_floresta.named_steps["classificador"],
    RandomForestClassifier
)

# Verificar que ainda nao foram treinados
assert not hasattr(
    pipeline_logistica.named_steps["classificador"],
    "coef_"
)

assert not hasattr(
    pipeline_floresta.named_steps["classificador"],
    "estimators_"
)

print("Pipeline da Regressao Logistica: OK")
print("Pipeline do Random Forest: OK")
print("Classificadores ainda nao treinados: OK")

print("\nEstruturas validadas com sucesso!")


=== VALIDACAO DOS PIPELINES ===
Pipeline da Regressao Logistica: OK
Pipeline do Random Forest: OK
Classificadores ainda nao treinados: OK

Estruturas validadas com sucesso!


=== VALIDACAO DOS PIPELINES ===

Pipeline da Regressao Logistica: OK
Pipeline do Random Forest: OK
Classificadores ainda nao treinados: OK

Estruturas validadas com sucesso!